<a href="https://colab.research.google.com/drive/1b2UC4fACgoVVdbrWNhVxlsC8jO8eGA1Q?usp=sharing" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

![CuratorKIT](CuratorKIT_logo-TP.png)

---

# Adaptive recovery

**Task:** `preference` &nbsp;|&nbsp; **Gates:** HallucinationGate + RewardGate &nbsp;|&nbsp; **Recovery:** Inline probe + post-pipeline refiner

> **Prerequisite:** This notebook needs a corpus JSONL as input. Either:
> - Run notebook 01 (Synthetic generation) first and set `CORPUS_PATH` to its output, or
> - Replace `CORPUS_PATH` with a HuggingFace dataset name (e.g. `"tatsu-lab/alpaca"`) to run standalone.

This notebook demonstrates the full self-healing pipeline. When gates reject samples,
two recovery layers attempt to fix them rather than discarding:

### Recovery layers (from PHASE1_PIPELINE.md)

```
Gate N
  ├─► passed
  └─► rejected ──► probe.diagnose_batch()
                     ├─► recovered  ──► merged into passed ──► Gate N+1
                     └─► still rejected ──► final rejected list

Post-pipeline (curator.run):
  └─► RewardRefiner.refine(reward_gate_rejects)
        ├─► recovered  ──► result.passed
        └─► still rejected ──► result.rejected
```

| Layer | When | Fires on |
|-------|------|---------|
| DiagnosticProbe (HallucinationGate) | Inline, before RewardGate | Grounding failures |
| DiagnosticProbe (RewardGate) | Inline, before next gate | Quality failures |
| RewardRefiner | Post-pipeline, after all gates | Remaining RewardGate rejects |

In [1]:
# ═══════════════════════════════════════════════════════════════════════════
# 0. Setup — clone + install CuratorKIT (run once, then comment out)
# Prerequisite: SSH key added to your GitHub account with repo access.
# ═══════════════════════════════════════════════════════════════════════════
from pathlib import Path

repo_dir = Path.home() / "CuratorKIT"
if not repo_dir.exists():
    !git clone https://github.com/Lexsi-Labs/CuratorKIT.git {repo_dir}

!pip install -e "{repo_dir}[generation,embedding,hf,parquet]" -q

print(f"CuratorKIT installed from {repo_dir}")

# !pip install "/content/curatorkit.tar.gz[generation,embedding,hf,parquet,pdf-gpu]" nest-asyncio -q

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.7/42.7 kB 1.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.4/68.4 kB 6.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.5/43.5 kB 4.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 162.6/162.6 kB 8.6 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 4.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.5/48.5 kB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.3/15.3 MB 99.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 87.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 786.6/786.6 kB 50.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.1/278.1 kB 30.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━

## 1. Configuration

In [3]:
MODEL    = "openai/Qwen/Qwen3-8B"
JUDGE    = "openai/Qwen/Qwen3-8B"
API_BASE = " "

In [4]:
from pathlib import Path
from curatorkit import Curator, CuratorConfig

# ── Source ────────────────────────────────────────────────────────────────
# Option A: Use output from a prior run (update path to your corpus JSONL)
# CORPUS_PATH = "output/synthetic_generation_qa/sft_alpaca.jsonl"
#
# Option B: Use a HuggingFace dataset directly (standalone, no prior run needed)
CORPUS_PATH = "tatsu-lab/alpaca"

# ── Pipeline params ───────────────────────────────────────────────────────
MAX_SAMPLES   = 50    # small for smoke test; increase for production
CONCURRENCY   = 32
OUTPUT_DIR    = Path("output/adaptive_recovery")

print(f"Source      : {CORPUS_PATH}")
print(f"Model       : {MODEL}")
print(f"Max samples : {MAX_SAMPLES}")
print(f"Output      : {OUTPUT_DIR}/")

Source      : tatsu-lab/alpaca
Model       : openai/Qwen/Qwen3-8B
Max samples : 50
Output      : output/adaptive_recovery/


## 2. Run pipeline with inline recovery

In [8]:
api_base = API_BASE if API_BASE else None

config = CuratorConfig(
    # ── Source ────────────────────────────────────────────────────────────
    dataset={"name": CORPUS_PATH, "max_samples": MAX_SAMPLES},
    max_tokens=4096,

    # ── Phase 0 ───────────────────────────────────────────────────────────
    dedup="minhash",
    clean=True,

    # ── Generator LLM (thinking ON for quality responses) ──────────────────
    llm_model=MODEL,
    llm_api_base=api_base,
    llm_max_tokens=2048,
    llm_concurrency=CONCURRENCY,
    generation_concurrency=CONCURRENCY,
    llm_api_key=" ",
    llm_extra_body={"chat_template_kwargs": {"enable_thinking": True}},

    # ── Judge LLM (thinking OFF for deterministic gate decisions) ──────────
    judge_llm_model=JUDGE,
    judge_llm_api_base=api_base,
    judge_llm_temperature=0.1,
    judge_concurrency=CONCURRENCY,
    judge_llm_extra_body={"chat_template_kwargs": {"enable_thinking": False}},

    # ── Preference generation ──────────────────────────────────────────────
    generation_task="preference",
    num_questions=1,

    # ── Gates (both active) ────────────────────────────────────────────────
    hallucination_threshold=0.7,
    reward_threshold=0.75,
    reward_dimensions=["helpfulness", "honesty", "instruction_following", "depth"],

    # ── Adaptive recovery layers ───────────────────────────────────────────
    enable_diagnostic_probe=True,            # inline: probe recovers near-miss rejects
    enable_reward_refiner=True,               # post-pipeline: refiner rewrites weak answers
    probe_temperatures=[0.3, 0.5],           # temperature sweep points
    probe_score_split=0.5,                   # routing threshold for low vs high grounding

    # ── Export ────────────────────────────────────────────────────────────
    export_formats=["dpo"],
    output_dir=OUTPUT_DIR,
)

print("Running pipeline with DiagnosticProbe + RewardRefiner...")
result = Curator(config).run()
result.print_summary()

Running pipeline with DiagnosticProbe + RewardRefiner...


RewardGate: 100%|██████████| 37/37 [00:22<00:00,  1.66sample/s, passed=15, rejected=22]

────────────────────────────────────────────
  passed   :       44
  rejected :       53
  time     :   303.1s
  output   : output/adaptive_recovery
────────────────────────────────────────────


## 3. Recovery statistics

In [9]:
# ── Diagnostic probe stats ────────────────────────────────────────────────
if result.diagnostics:
    d = result.diagnostics.to_dict()
    print("DiagnosticProbe results:")
    print(f"  total_diagnosed   : {d.get('total_diagnosed')}")
    print(f"  probe_recovered   : {d.get('probe_recovered')}  (inline recovery)")
    print(f"  probe_recovery_pct: {d.get('probe_recovery_pct', 0):.1%}")
    print(f"  total_probe_calls : {d.get('total_probe_calls')}  (extra LLM calls)")
    print(f"\nFailure mode distribution:")
    for mode, count in d.get('mode_counts', {}).items():
        print(f"  {mode:<30} {count:>4}")
else:
    print("No diagnostics collected — probe may not have fired on any rejects.")

DiagnosticProbe results:
  total_diagnosed   : 67
  probe_recovered   : 47  (inline recovery)
  probe_recovery_pct: 70.2%
  total_probe_calls : 159  (extra LLM calls)

Failure mode distribution:
  generator_parametric             29
  unknown                           9
  threshold_marginal               14
  instruction_quality               3
  domain_mismatch                   1
  response_quality                  5
  source_ambiguous                  6


In [13]:
# ── Sample preview ────────────────────────────────────────────────────────
if result.passed:
    s = result.passed[0]
    print(f"Passed pairs: {len(result.passed):,}")
    print(f"\nSample DPO pair:")
    print(f"  instruction : {s.instruction[:240]}")
    print(f"  chosen      : {s.chosen[:512]}")
    print(f"  rejected    : {s.rejected[:512]}")

    # Check if any passed sample was probe-recovered
    recovered = [
        s for s in result.passed
        if any(rec.step_name == "DiagnosticProbe" for rec in s.provenance_chain)
    ]
    print(f"\nProbe-recovered samples in final output: {len(recovered)}")

Passed pairs: 44

Sample DPO pair:
  instruction : What are the three primary colors?
  chosen      : The three primary colors are red, blue, and yellow.
  rejected    : ) based on an instruction and source passage.
   - **Instruction:** "What are the three primary colors?"
   - **Source Passage:** "The three primary colors are red, blue, and yellow."
   - **Requirements for Chosen:** Thorough, accurate, addresses all aspects with specifics.
   - **Requirements for Rejected:** Use EXACTLY ONE degradation pattern from:
     - Omit the most important specific detail
     - Use vague language where chosen is concrete
     - Answer only part of a multi-part question
   - **Cons

Probe-recovered samples in final output: 32


## 4. Probe call budget (from PHASE1_PIPELINE.md)

| Scenario | LLM calls |
|----------|----------|
| `rejected_above_threshold` (contrast failure) | 0 — early exit |
| Temperature sweep resolves at T=0.3 | 1 |
| Temperature sweep resolves at T=0.5 | 2 |
| Strict grounding resolves immediately | 1 |
| Full sweep + all prompt variants | 5 (worst case) |
| All probes fail → UNKNOWN | 5 + 0 recovery |

### Variations to try
- **Wider temperature sweep:** `probe_temperatures=[0.2, 0.4, 0.6, 0.8]` — tests more sampling regimes
- **Higher score split:** `probe_score_split=0.6` — routes more samples to grounding-first path
- **Custom probe templates:** `probe_extra_templates={"strict_grounding": "..."}` — domain-specific grounding prompts
- **Disable refiner:** comment out `enable_reward_refiner=True` — see how much the refiner contributes vs. inline probe alone